# 🏆 第6步：WideResNet —— 用小图榨出更多信息

## 为什么加宽比加深更适合 CIFAR-100？

CIFAR 图片只有 32×32，经过几次 stride=2 后特征图就变成 4×4 甚至 2×2。
继续加深（ResNet-50、ResNet-101）没有意义——4×4 的图上堆再多层也提取不出新东西。

**加宽**是更好的方向：通道数多 = 每层同时看更多种特征（纹理、形状、颜色），
配合你的强数据增强，模型容量才能充分发挥。

```
ResNet-18 (你现在的):      WideResNet-28-8 (新):
┌──────────────┐          ┌─────────────────┐
│  64 → 128    │          │  128 → 256      │
│  128 → 256   │          │  256 → 512      │
│  256 → 512   │          │  512 → 1024     │
│  512 → 512   │          │  1024 → 1024    │
│  11M 参数    │          │  ~24M 参数       │
└──────────────┘          └─────────────────┘
   72.6%                     预期 78%+
```

> 🎯 目标：72.6% → 78%+

---
## Cell 1：导入库 & 加载数据

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import time
from collections import Counter

# ===== 设备 =====
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 CUDA: {torch.cuda.get_device_name(0)}")
    print(f"   GPU 数: {torch.cuda.device_count()}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Apple MPS")
else:
    device = torch.device("cpu")
    print("🐢 CPU")

# ===== 数据增强（保持第5步的配置）=====
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

# ===== 加载数据 =====
full_ds = ImageFolder(root='./CIFAR-100数据集/CIFAR-100数据集', transform=transform_train)
targets = full_ds.targets
train_idx, test_idx = [], []
for c in range(100):
    ci = [i for i, t in enumerate(targets) if t == c]
    train_idx.extend(ci[:500]); test_idx.extend(ci[500:])

train_ds = Subset(full_ds, train_idx)
full_test = ImageFolder(root='./CIFAR-100数据集/CIFAR-100数据集', transform=transform_test)
test_ds = Subset(full_test, test_idx)

batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=8, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                         num_workers=4, pin_memory=True)
print(f"训练: {len(train_ds)}, 测试: {len(test_ds)}, batch={batch_size}")

---
## Cell 2：WideResNet 的基本积木 —— WideBasicBlock

和 ResNet 的 BasicBlock 结构一样，区别在于：

| | BasicBlock | WideBasicBlock |
|------|-----------|---------------|
| 通道数 | 64→128→256→512 | **128→256→512→1024** |
| Dropout | 没有 | **卷积层之间有 Dropout** |

Dropout 加在卷积之间是 WideResNet 原论文的关键设计——用 Dropout 替代部分正则化。

In [ ]:
class WideBasicBlock(nn.Module):
    """WideResNet 的基本积木：BN→ReLU→Conv → BN→ReLU→Conv + shortcut"
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, dropout=0.1):
        super().__init__()

        # 主路径（注意：WideResNet 的 BN+ReLU 在 Conv 之前，和标准 ResNet 相反）
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(dropout)  # ← 卷积之间加 Dropout
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)

        # 跳跃连接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False)

    def forward(self, x):
        out = torch.relu(self.bn1(x))              # BN → ReLU
        out = self.conv1(out)                       # Conv1
        out = self.dropout(torch.relu(self.bn2(out)))  # BN → ReLU → Dropout
        out = self.conv2(out)                       # Conv2
        out += self.shortcut(x)                     # + shortcut
        return out

# 测试
b = WideBasicBlock(64, 128, stride=2)
x = torch.randn(4, 64, 32, 32)
out = b(x)
print(f"WideBasicBlock: {x.shape} → {out.shape}")
print(f"  stride=2 所以尺寸减半，通道 64→128")

---
## Cell 3：组装 WideResNet-28-8

```
输入 [B, 3, 32, 32]
  │
  ├─ Conv2d(3→16, 3×3)               → [B, 16, 32, 32]
  │
  ├─ Group1: WideBasicBlock×4         → [B, 128,  32, 32]  (16→128, stride=1)
  ├─ Group2: WideBasicBlock×4         → [B, 256,  16, 16]  (stride=2)
  ├─ Group3: WideBasicBlock×4         → [B, 512,  8, 8 ]   (stride=2)
  │
  ├─ BN + ReLU                        → [B, 512,  8, 8 ]
  ├─ AdaptiveAvgPool2d(1)             → [B, 512,  1, 1 ]
  ├─ Flatten                          → [B, 512]
  ├─ Linear(512→100)                  → [B, 100]
```

**widen_factor=8**：每个 Group 的通道数翻 8 倍。

In [ ]:
class WideResNet(nn.Module):
    """WideResNet for CIFAR-100
    Args:
        depth: 总层数，必须是 6n+4（n = 每组的 block 数）
        widen_factor: 通道扩大倍数
        dropout: Dropout 概率
    """
    def __init__(self, depth=28, widen_factor=8, dropout=0.3, num_classes=100):
        super().__init__()

        assert (depth - 4) % 6 == 0, f"depth 必须是 6n+4, 你给的 {depth}"
        n = (depth - 4) // 6  # 每组的 block 数

        # 初始通道
        base_ch = 16
        self.in_channels = base_ch

        # 初始卷积
        self.conv1 = nn.Conv2d(3, base_ch, 3, stride=1, padding=1, bias=False)

        # 三个 Group
        self.group1 = self._make_group(base_ch * widen_factor, n, stride=1, dropout=dropout)
        #                           16*8=128
        self.group2 = self._make_group(base_ch * widen_factor * 2, n, stride=2, dropout=dropout)
        #                           32*8=256
        self.group3 = self._make_group(base_ch * widen_factor * 4, n, stride=2, dropout=dropout)
        #                           64*8=512

        # 最后的 BN + 分类头
        self.bn = nn.BatchNorm2d(base_ch * widen_factor * 4)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(base_ch * widen_factor * 4, num_classes)

        # 初始化
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_group(self, out_channels, num_blocks, stride, dropout):
        """创建一组 WideBasicBlock"""
        layers = []
        # 第一个 Block 负责通道和尺寸变化
        layers.append(WideBasicBlock(self.in_channels, out_channels, stride, dropout))
        self.in_channels = out_channels
        # 后续 Block 保持通道和尺寸
        for _ in range(1, num_blocks):
            layers.append(WideBasicBlock(self.in_channels, out_channels, 1, dropout))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.group1(x)
        x = self.group2(x)
        x = self.group3(x)
        x = torch.relu(self.bn(x))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


# 创建模型
model = WideResNet(depth=28, widen_factor=8, dropout=0.3, num_classes=100)

# 双卡并行
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

# 测试
x = torch.randn(4, 3, 32, 32).to(device)
out = model(x)
print(f"输入: {list(x.shape)}  →  输出: {list(out.shape)}")

n = sum(p.numel() for p in model.parameters())
print(f"参数量: {n:,} ({n/1e6:.1f}M)")
print(f"  ResNet-18: 11.2M → WideResNet-28-8: {n/1e6:.1f}M")

---
## Cell 4：训练配置

In [ ]:
# ===== 损失函数 =====
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ===== 优化器 =====
optimizer = optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True
)

# ===== 余弦退火（简单版，60 epoch）=====
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-5
)

print(f"损失函数: CrossEntropyLoss + Label Smoothing(0.1)")
print(f"优化器:   SGD (lr=0.1, momentum=0.9, nesterov=True, wd=5e-4)")
print(f"调度器:   CosineAnnealing (T_max=100, 0.1 → 1e-5)")

---
## Cell 5：训练函数

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch"""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    """验证"""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    return running_loss / len(loader), 100. * correct / total

print("✅ 训练/验证函数已定义")

---
## Cell 6：开始训练 🔥

WideResNet-28-8, 60 epoch, 3090 上约 35~40 分钟。

In [ ]:
num_epochs = 100
best_acc = 0.0
history = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[], 'lr':[]}

print(f"训练 {num_epochs} epoch | WideResNet-28-8 + Cosine")
print(f"设备: {device}")
if torch.cuda.is_available():
    print(f"GPU 数: {torch.cuda.device_count()}")
print("=" * 60)

t0 = time.time()
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    test_loss, test_acc = validate(model, test_loader, criterion, device)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['lr'].append(current_lr)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_wideresnet.pth')

    print(f"Epoch {epoch+1:3d}/{num_epochs} | "
          f"Train: {train_loss:.3f} {train_acc:.1f}% | "
          f"Test: {test_loss:.3f} {test_acc:.1f}% | "
          f"lr={current_lr:.1e} | 最佳: {best_acc:.1f}%")

elapsed = time.time() - t0
print("=" * 60)
print(f"完成！耗时 {elapsed//60:.0f}分{elapsed%60:.0f}秒")
print(f"最佳测试准确率: {best_acc:.2f}%")

---
## Cell 7：训练曲线

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train', color='#2196F3')
axes[0].plot(history['test_loss'], label='Test', color='#FF5722')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss 曲线 (WideResNet-28-8)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', color='#2196F3')
axes[1].plot(history['test_acc'], label='Test', color='#FF5722')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy 曲线'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['lr'], color='#4CAF50')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('学习率 (Cosine)'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wideresnet_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ 已保存 wideresnet_curves.png")

---
## Cell 8：最终评估

In [ ]:
model.load_state_dict(torch.load('best_wideresnet.pth', weights_only=True))
model = model.to(device)
test_loss, test_acc = validate(model, test_loader, criterion, device)

print(f"WideResNet-28-8 最终结果: Loss={test_loss:.4f}, Acc={test_acc:.2f}%")
print(f"ResNet-18 (第5步):        Acc=72.6%")
print(f"提升:                    +{test_acc-72.6:.1f}% 🚀")

---
## 🎉 第6步完成！

| ✅ | 新知识点 | 一句话 |
|----|--------|--------|
| ① | **加宽 vs 加深** | 32×32 小图上，加宽通道比加深层数更有效 |
| ② | **WideBasicBlock** | BN→ReLU→Conv→BN→ReLU→Dropout→Conv + shortcut |
| ③ | **Dropout in Conv** | 卷积层之间加 Dropout（不是池化后），正则化更强 |
| ④ | **widen_factor** | 控制通道宽度的倍数，8~10 是常用值 |

### 全系列对比

| | ResNet-18 | WideResNet-28-8 |
|------|-----------|----------------|
| 参数量 | 11.2M | ~24M |
| 通道数 | 64→128→256→512 | 128→256→512→1024 |
| 准确率 | 72.6% | 预期 **78%+** |
